# EfficientNet-B0/B1/B2/B3/B4 Controlled Ablation

This notebook compares EfficientNet-B0, B1, B2, B3, and B4 for the same KL-grade
task. The goal is to test whether B4 is unnecessarily large for this approximately
5,000-image dataset, while measuring both predictive validation metrics and native
CAM localization.

Every variant uses the same validation split, laterality normalization, CLAHE,
augmentation, full inverse-frequency sampler, CE loss, staged unfreezing schedule,
selection score, and native-CAM head. Only the EfficientNet scale changes.

The test split is intentionally not read. This is a validation ablation. After the
best scale is selected, run its standalone production notebook once for the locked
holdout evaluation and the 50-per-grade CAM audit.


## Run Instructions

Run every cell from top to bottom on a fresh runtime. Five models are trained
sequentially, so an A100 is recommended. The physical micro-batches are fixed for a
15-16 GiB GPU and gradient accumulation preserves an effective batch size of 48:

| Variant | Physical batch | Accumulation | Effective batch |
| --- | ---: | ---: | ---: |
| B0 | 24 | 2 | 48 |
| B1 | 24 | 2 | 48 |
| B2 | 16 | 3 | 48 |
| B3 | 12 | 4 | 48 |
| B4 | 8 | 6 | 48 |

The output contains one timestamped directory per variant, validation metrics, CAM
geometry/occlusion summaries, comparison CSVs, and a markdown report. It does not
promote a model or evaluate the test set.


In [1]:
import csv
import gc
import hashlib
import json
import random
import subprocess
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms
import tqdm
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    cohen_kappa_score,
    precision_recall_fscore_support,
    recall_score,
    roc_auc_score,
)
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import models

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not running in Colab; Drive mount skipped.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory GiB: {torch.cuda.get_device_properties(0).total_memory / 2**30:.2f}")
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

SEED = 42
NUM_WORKERS = 4
STAGE_EPOCHS = (5, 15, 10)
CAM_AUDIT_PER_GRADE = 50
OCCLUSION_GRID = 4

VARIANT_SPECS = [
    {
        "name": "efficientnet_b0",
        "builder": models.efficientnet_b0,
        "weights": models.EfficientNet_B0_Weights.DEFAULT,
        "physical_batch_size": 24,
        "accumulation_steps": 2,
    },
    {
        "name": "efficientnet_b1",
        "builder": models.efficientnet_b1,
        "weights": models.EfficientNet_B1_Weights.DEFAULT,
        "physical_batch_size": 24,
        "accumulation_steps": 2,
    },
    {
        "name": "efficientnet_b2",
        "builder": models.efficientnet_b2,
        "weights": models.EfficientNet_B2_Weights.DEFAULT,
        "physical_batch_size": 16,
        "accumulation_steps": 3,
    },
    {
        "name": "efficientnet_b3",
        "builder": models.efficientnet_b3,
        "weights": models.EfficientNet_B3_Weights.DEFAULT,
        "physical_batch_size": 12,
        "accumulation_steps": 4,
    },
    {
        "name": "efficientnet_b4",
        "builder": models.efficientnet_b4,
        "weights": models.EfficientNet_B4_Weights.DEFAULT,
        "physical_batch_size": 8,
        "accumulation_steps": 6,
    },
]

for spec in VARIANT_SPECS:
    spec["effective_batch_size"] = (
        spec["physical_batch_size"] * spec["accumulation_steps"]
    )

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

dataset_zip = Path(
    "/content/drive/MyDrive/Datasets/kaggle_knee_osteoarthritis.zip"
)
if dataset_zip.exists():
    subprocess.run(
        ["unzip", "-q", "-o", str(dataset_zip), "-d", "/content/Datasets"],
        check=True,
    )
DATASET_ROOT = Path("/content/Datasets/kaggle_knee_osteoarthritis")
RUN_TIMESTAMP = datetime.now(timezone.utc).strftime(
    "%Y-%m-%d_%H-%M-%S_%f_UTC"
)
RUN_ROOT = (
    Path("/content/drive/MyDrive/Models/efficientnet_scale_ablations")
    / f"{RUN_TIMESTAMP}_b0_b1_b2_b3_b4"
)
RUN_ROOT.mkdir(parents=True, exist_ok=False)
print(f"Ablation root: {RUN_ROOT}")


Mounted at /content/drive
Device: cuda
GPU: Tesla T4
GPU memory GiB: 14.56
Ablation root: /content/drive/MyDrive/Models/efficientnet_scale_ablations/2026-07-23_16-08-50_642801_UTC_b0_b1_b2_b3_b4


## Data Preparation

The dataset already contains knee crops, so YOLO is not used in this training
ablation. Right knees are mirrored into the same canonical orientation as the
training reference. Duplicate validation images are removed by SHA-256.


In [2]:
class SquarePad:
    def __call__(self, image):
        height, width = image.shape[:2]
        side = max(height, width)
        top = (side - height) // 2
        bottom = side - height - top
        left = (side - width) // 2
        right = side - width - left
        return cv2.copyMakeBorder(
            image, top, bottom, left, right,
            cv2.BORDER_CONSTANT, value=[0, 0, 0],
        )


class CLAHE:
    def __call__(self, image):
        lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
        lightness, channel_a, channel_b = cv2.split(lab)
        lightness = cv2.createCLAHE(
            clipLimit=2.0, tileGridSize=(8, 8)
        ).apply(lightness)
        return cv2.cvtColor(
            cv2.merge((lightness, channel_a, channel_b)),
            cv2.COLOR_LAB2RGB,
        )


train_transform = transforms.Compose([
    SquarePad(),
    CLAHE(),
    transforms.ToPILImage(),
    transforms.RandomRotation(5),
    transforms.ColorJitter(brightness=0.08, contrast=0.08),
    transforms.Resize((400, 400)),
    transforms.RandomCrop(384),
    transforms.ToTensor(),
    transforms.RandomErasing(
        p=0.10, scale=(0.02, 0.05), ratio=(0.5, 2.0), value=0
    ),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

evaluation_transform = transforms.Compose([
    SquarePad(),
    CLAHE(),
    transforms.ToPILImage(),
    transforms.Resize((400, 400)),
    transforms.CenterCrop(384),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])


def file_digest(path):
    digest = hashlib.sha256()
    with open(path, "rb") as image_file:
        for chunk in iter(lambda: image_file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


class KneeDataset(Dataset):
    def __init__(self, split, transform, excluded_hashes=None):
        self.transform = transform
        self.paths = []
        self.labels = []
        excluded_hashes = excluded_hashes or set()
        retained_hashes = set()
        for grade in range(5):
            grade_dir = DATASET_ROOT / split / str(grade)
            if not grade_dir.is_dir():
                raise FileNotFoundError(f"Missing dataset directory: {grade_dir}")
            for path in sorted(grade_dir.iterdir()):
                if path.suffix.lower() not in {".png", ".jpg", ".jpeg"}:
                    continue
                digest = file_digest(path)
                if digest in excluded_hashes or digest in retained_hashes:
                    continue
                retained_hashes.add(digest)
                self.paths.append(str(path))
                self.labels.append(grade)
        self.image_hashes = retained_hashes
        print(f"{split}: {len(self.paths)} unique images")

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        path = self.paths[index]
        image = cv2.imread(path)
        if image is None:
            raise IOError(f"Could not read image: {path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        if Path(path).stem.upper().endswith("R"):
            image = np.ascontiguousarray(image[:, ::-1])
        return self.transform(image), self.labels[index], path


train_dataset = KneeDataset("train", train_transform)
validation_dataset = KneeDataset(
    "val", evaluation_transform, train_dataset.image_hashes
)
class_counts = Counter(train_dataset.labels)
sample_weights = [1.0 / class_counts[label] for label in train_dataset.labels]
print("Class counts:", dict(sorted(class_counts.items())))


train: 5778 unique images
val: 826 unique images
Class counts: {0: 2286, 1: 1046, 2: 1516, 3: 757, 4: 173}


## Model and Training Helpers

Each variant uses the same five-channel 1x1 class head. Logits are the spatial means
of the class maps, so native CAM is faithful to the model output. No EMA or
multiscale fusion is used.


In [3]:
class EfficientNetNativeCAM(nn.Module):
    def __init__(self, spec):
        super().__init__()
        network = spec["builder"](weights=spec["weights"])
        self.features = network.features
        self.class_conv = nn.Conv2d(
            network.classifier[1].in_features, 5, kernel_size=1
        )

    def head_parameters(self):
        return self.class_conv.parameters()

    def class_maps(self, images):
        return self.class_conv(self.features(images))

    def forward(self, images):
        return self.class_maps(images).mean(dim=(2, 3))

    @staticmethod
    def native_cam_from_maps(class_maps, class_index, output_size):
        selected = F.relu(class_maps[:, class_index : class_index + 1])
        selected = F.interpolate(
            selected, size=output_size, mode="bilinear", align_corners=False
        )[0, 0]
        return selected / selected.max().clamp_min(1e-8)

    def native_cam(self, images, class_index):
        with torch.no_grad():
            maps = self.class_maps(images)
            logits = maps.mean(dim=(2, 3))
            cam = self.native_cam_from_maps(
                maps, int(class_index), images.shape[-2:]
            )
        return cam, logits

    def freeze_backbone(self):
        for parameter in self.features.parameters():
            parameter.requires_grad = False
        for parameter in self.class_conv.parameters():
            parameter.requires_grad = True

    def unfreeze_final_stages(self):
        for parameter in self.parameters():
            parameter.requires_grad = False
        for block in self.features[6:]:
            for parameter in block.parameters():
                parameter.requires_grad = True
        for parameter in self.class_conv.parameters():
            parameter.requires_grad = True

    def unfreeze_all(self):
        for parameter in self.parameters():
            parameter.requires_grad = True


def keep_frozen_batch_norm_eval(model):
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d) and not any(
            parameter.requires_grad for parameter in module.parameters()
        ):
            module.eval()


def make_loaders(spec):
    generator = torch.Generator().manual_seed(SEED)
    sampler = WeightedRandomSampler(
        sample_weights,
        num_samples=len(sample_weights),
        replacement=True,
        generator=generator,
    )
    train_loader = DataLoader(
        train_dataset,
        batch_size=spec["physical_batch_size"],
        sampler=sampler,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True,
        generator=generator,
    )
    validation_loader = DataLoader(
        validation_dataset,
        batch_size=spec["physical_batch_size"],
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True,
    )
    return train_loader, validation_loader


def calculate_metrics(labels, predictions, probabilities):
    labels = np.asarray(labels)
    predictions = np.asarray(predictions)
    probabilities = np.asarray(probabilities)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, labels=[0, 1, 2, 3, 4],
        average="macro", zero_division=0,
    )
    one_hot = np.eye(5)[labels]
    metrics = {
        "accuracy": float(accuracy_score(labels, predictions)),
        "qwk": float(cohen_kappa_score(labels, predictions, weights="quadratic")),
        "macro_precision": float(precision),
        "macro_recall": float(recall),
        "macro_f1": float(f1),
        "grade1_recall": float(
            recall_score(labels == 1, predictions == 1, zero_division=0)
        ),
        "macro_ap": float(
            average_precision_score(one_hot, probabilities, average="macro")
        ),
        "macro_auc": float(
            roc_auc_score(one_hot, probabilities, average="macro")
        ),
    }
    metrics["selection_score"] = float(
        0.40 * metrics["qwk"]
        + 0.20 * metrics["macro_f1"]
        + 0.10 * metrics["macro_recall"]
        + 0.10 * metrics["grade1_recall"]
        + 0.15 * metrics["macro_ap"]
        + 0.05 * metrics["macro_auc"]
    )
    return metrics


def evaluate(model, loader):
    model.eval()
    labels_all, predictions_all, probabilities_all = [], [], []
    with torch.no_grad():
        for images, labels, _ in loader:
            images = images.to(device, non_blocking=True)
            logits = model(images)
            probabilities = F.softmax(logits.float(), dim=1)
            labels_all.extend(labels.numpy())
            predictions_all.extend(probabilities.argmax(1).cpu().numpy())
            probabilities_all.extend(probabilities.cpu().numpy())
    return calculate_metrics(labels_all, predictions_all, probabilities_all)


def train_one_epoch(model, loader, optimizer, scaler, accumulation_steps):
    model.train()
    keep_frozen_batch_norm_eval(model)
    running_loss = 0.0
    number_of_batches = len(loader)
    optimizer.zero_grad(set_to_none=True)
    for batch_index, (images, labels, _) in enumerate(loader):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        group_start = (batch_index // accumulation_steps) * accumulation_steps
        group_size = min(accumulation_steps, number_of_batches - group_start)
        with torch.amp.autocast(
            device_type=device.type, enabled=device.type == "cuda"
        ):
            loss = F.cross_entropy(model(images), labels)
            backward_loss = loss / group_size
        scaler.scale(backward_loss).backward()
        boundary = (
            (batch_index + 1) % accumulation_steps == 0
            or batch_index + 1 == number_of_batches
        )
        if boundary:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
        running_loss += loss.item() * labels.size(0)
    return running_loss / len(loader.dataset)


def load_checkpoint(path):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


## Train All Five Scales

Each scale receives its own timestamped directory. Checkpoints are selected only by
the validation composite; the test split remains locked for the later production
run of the selected scale.


In [4]:
def train_variant(spec):
    variant_start = time.time()
    variant_dir = RUN_ROOT / spec["name"]
    variant_dir.mkdir(parents=True, exist_ok=False)
    train_loader, validation_loader = make_loaders(spec)
    model = EfficientNetNativeCAM(spec).to(device)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    history = []
    best_score = -float("inf")
    best_path = variant_dir / "best_model.pth"
    last_path = variant_dir / "last_model.pth"
    global_epoch = 0

    for stage_name, stage_epochs in zip(
        ("warmup", "coarse", "finetune"), STAGE_EPOCHS
    ):
        if stage_name == "warmup":
            model.freeze_backbone()
            optimizer = optim.AdamW(
                model.head_parameters(), lr=3e-4, weight_decay=1e-4
            )
            scheduler = None
        elif stage_name == "coarse":
            model.unfreeze_final_stages()
            optimizer = optim.AdamW(
                [
                    {
                        "params": [
                            parameter for parameter in model.features.parameters()
                            if parameter.requires_grad
                        ],
                        "lr": 3e-5,
                    },
                    {"params": model.head_parameters(), "lr": 3e-4},
                ],
                weight_decay=1e-4,
            )
            scheduler = optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=stage_epochs, eta_min=1e-7
            )
        else:
            model.load_state_dict(load_checkpoint(variant_dir / "stage2_best_model.pth")["model_state_dict"])
            model.unfreeze_all()
            optimizer = optim.AdamW(
                model.parameters(), lr=1e-5, weight_decay=1e-3
            )
            scheduler = optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=stage_epochs, eta_min=1e-7
            )

        best_stage2_score = -float("inf")
        for stage_epoch in range(stage_epochs):
            global_epoch += 1
            train_loss = train_one_epoch(
                model, train_loader, optimizer, scaler, spec["accumulation_steps"]
            )
            validation_metrics = evaluate(model, validation_loader)
            history.append({
                "epoch": global_epoch,
                "stage": stage_name,
                "train_loss": train_loss,
                **validation_metrics,
            })
            print(
                f"{spec['name']} | epoch {global_epoch:02d} | {stage_name} | "
                f"QWK={validation_metrics['qwk']:.4f} | "
                f"F1={validation_metrics['macro_f1']:.4f} | "
                f"selection={validation_metrics['selection_score']:.4f}"
            )
            if scheduler is not None:
                scheduler.step()
            payload = {
                "model_state_dict": model.state_dict(),
                "epoch": global_epoch,
                "stage": stage_name,
                "architecture": "efficientnet_scale_final_native_cam_ce",
                "model_name": spec["name"],
                "loss_type": "ce",
                "validation_metrics": validation_metrics,
                "history": history,
                "run_timestamp": RUN_TIMESTAMP,
                "experimental_config": {
                    "input_resize": 400,
                    "input_crop": 384,
                    "physical_batch_size": spec["physical_batch_size"],
                    "gradient_accumulation_steps": spec["accumulation_steps"],
                    "effective_batch_size": spec["effective_batch_size"],
                    "sampler": "full_inverse_frequency",
                    "canonicalize_laterality": True,
                    "stage_epochs": list(STAGE_EPOCHS),
                    "native_cam": True,
                    "ema": False,
                },
            }
            if stage_name == "coarse" and validation_metrics["selection_score"] > best_stage2_score:
                best_stage2_score = validation_metrics["selection_score"]
                torch.save(payload, variant_dir / "stage2_best_model.pth")
            if stage_name == "finetune" and validation_metrics["selection_score"] > best_score:
                best_score = validation_metrics["selection_score"]
                torch.save(payload, best_path)
            torch.save(payload, last_path)

    with open(variant_dir / "epoch_metrics.csv", "w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=history[0].keys())
        writer.writeheader()
        writer.writerows(history)
    selected = load_checkpoint(best_path)
    elapsed_minutes = (time.time() - variant_start) / 60.0
    manifest = {
        "variant": spec["name"],
        "run_timestamp": RUN_TIMESTAMP,
        "best_epoch": selected["epoch"],
        "validation_metrics": selected["validation_metrics"],
        "elapsed_minutes": elapsed_minutes,
        "checkpoint": str(best_path),
        "config": {
            "variant": spec["name"],
            "physical_batch_size": spec["physical_batch_size"],
            "gradient_accumulation_steps": spec["accumulation_steps"],
            "effective_batch_size": spec["effective_batch_size"],
            "input_resize": 400,
            "input_crop": 384,
            "sampler": "full_inverse_frequency",
            "stage_epochs": list(STAGE_EPOCHS),
            "loss": "cross_entropy",
            "native_cam": True,
            "ema": False,
        },
    }
    with open(variant_dir / "run_manifest.json", "w") as handle:
        json.dump(manifest, handle, indent=2)
    return manifest, variant_dir


trained = {}
for variant_spec in VARIANT_SPECS:
    manifest, variant_dir = train_variant(variant_spec)
    trained[variant_spec["name"]] = {
        "manifest": manifest,
        "directory": variant_dir,
    }
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 148MB/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


efficientnet_b0 | epoch 01 | warmup | QWK=0.4971 | F1=0.3060 | selection=0.4454
efficientnet_b0 | epoch 02 | warmup | QWK=0.4856 | F1=0.2362 | selection=0.3814
efficientnet_b0 | epoch 03 | warmup | QWK=0.5222 | F1=0.2901 | selection=0.4535
efficientnet_b0 | epoch 04 | warmup | QWK=0.4944 | F1=0.2729 | selection=0.4126
efficientnet_b0 | epoch 05 | warmup | QWK=0.5588 | F1=0.3234 | selection=0.4674
efficientnet_b0 | epoch 06 | coarse | QWK=0.6681 | F1=0.4795 | selection=0.5537
efficientnet_b0 | epoch 07 | coarse | QWK=0.6945 | F1=0.5113 | selection=0.5814
efficientnet_b0 | epoch 08 | coarse | QWK=0.7280 | F1=0.5888 | selection=0.6265
efficientnet_b0 | epoch 09 | coarse | QWK=0.7385 | F1=0.5899 | selection=0.6464
efficientnet_b0 | epoch 10 | coarse | QWK=0.7743 | F1=0.6317 | selection=0.6936
efficientnet_b0 | epoch 11 | coarse | QWK=0.7521 | F1=0.6256 | selection=0.6697
efficientnet_b0 | epoch 12 | coarse | QWK=0.7474 | F1=0.6098 | selection=0.6617
efficientnet_b0 | epoch 13 | coarse | QW

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


...

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Downloading: "https://download.pytorch.org/models/efficientnet_b4_rwightman-23ab8bcd.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b4_rwightman-23ab8bcd.pth


100%|██████████| 74.5M/74.5M [00:00<00:00, 174MB/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


efficientnet_b4 | epoch 01 | warmup | QWK=0.4810 | F1=0.3392 | selection=0.4487
efficientnet_b4 | epoch 02 | warmup | QWK=0.4903 | F1=0.3012 | selection=0.4534
efficientnet_b4 | epoch 03 | warmup | QWK=0.5222 | F1=0.3505 | selection=0.4798
efficientnet_b4 | epoch 04 | warmup | QWK=0.4612 | F1=0.2805 | selection=0.3961
efficientnet_b4 | epoch 05 | warmup | QWK=0.5679 | F1=0.3578 | selection=0.4949
efficientnet_b4 | epoch 06 | coarse | QWK=0.6187 | F1=0.4380 | selection=0.5163
efficientnet_b4 | epoch 07 | coarse | QWK=0.6586 | F1=0.4181 | selection=0.5268
efficientnet_b4 | epoch 08 | coarse | QWK=0.7066 | F1=0.4979 | selection=0.5767
efficientnet_b4 | epoch 09 | coarse | QWK=0.6625 | F1=0.4859 | selection=0.5569
efficientnet_b4 | epoch 10 | coarse | QWK=0.7428 | F1=0.5698 | selection=0.6275
efficientnet_b4 | epoch 11 | coarse | QWK=0.7540 | F1=0.5742 | selection=0.6507
efficientnet_b4 | epoch 12 | coarse | QWK=0.7542 | F1=0.5804 | selection=0.6408
efficientnet_b4 | epoch 13 | coarse | QW

: 

## Native-CAM Audit: 50 Cases Per Grade

The same 250 validation knees are audited for every scale. Geometry measures whether
the heatmap stays in the joint region rather than the border or lower tibia. A 4x4
mean-filled occlusion test measures whether high-energy cells cause larger drops in
the selected class probability.


In [ ]:
def cam_geometry(cam):
    height, width = cam.shape
    joint = np.zeros((height, width), dtype=bool)
    joint[int(0.28 * height):int(0.72 * height), int(0.06 * width):int(0.94 * width)] = True
    border = np.ones((height, width), dtype=bool)
    border[int(0.08 * height):int(0.92 * height), int(0.08 * width):int(0.92 * width)] = False
    lower_tibia = np.zeros((height, width), dtype=bool)
    lower_tibia[int(0.72 * height):, :] = True
    total = float(cam.sum()) + 1e-8
    peak = np.unravel_index(np.argmax(cam), cam.shape)
    return {
        "joint_energy": float(cam[joint].sum()) / total,
        "border_energy": float(cam[border].sum()) / total,
        "lower_tibia_energy": float(cam[lower_tibia].sum()) / total,
        "peak_inside_joint": int(joint[peak]),
    }


def occlusion_metrics(model, image, target_class, cam, physical_batch_size):
    height, width = image.shape[-2:]
    patch_height = height // OCCLUSION_GRID
    patch_width = width // OCCLUSION_GRID
    variants, energies = [], []
    for row in range(OCCLUSION_GRID):
        for column in range(OCCLUSION_GRID):
            y0, y1 = row * patch_height, height if row == OCCLUSION_GRID - 1 else (row + 1) * patch_height
            x0, x1 = column * patch_width, width if column == OCCLUSION_GRID - 1 else (column + 1) * patch_width
            variant = image.clone()
            variant[:, y0:y1, x0:x1] = 0.0
            variants.append(variant)
            energies.append(float(cam[y0:y1, x0:x1].sum()))
    with torch.no_grad():
        baseline = F.softmax(model(image[None]), dim=1)[0, target_class]
        probabilities = []
        for chunk in torch.stack(variants).split(physical_batch_size):
            probabilities.append(
                F.softmax(model(chunk.to(device)), dim=1)[:, target_class].cpu()
            )
    drops = (baseline.cpu() - torch.cat(probabilities)).numpy()
    energies = np.asarray(energies)
    correlation = 0.0 if np.std(energies) < 1e-8 or np.std(drops) < 1e-8 else float(np.corrcoef(energies, drops)[0, 1])
    top = np.argsort(energies)[-max(1, len(energies) // 4):]
    return {
        "occlusion_correlation": correlation,
        "top_cam_confidence_drop": float(np.mean(drops[top])),
    }


rng = np.random.default_rng(SEED)
audit_indices = []
validation_labels = np.asarray(validation_dataset.labels)
for grade in range(5):
    grade_indices = np.flatnonzero(validation_labels == grade)
    rng.shuffle(grade_indices)
    audit_indices.extend(grade_indices[:min(CAM_AUDIT_PER_GRADE, len(grade_indices))].tolist())

cam_summaries = []
for spec in VARIANT_SPECS:
    entry = trained[spec["name"]]
    model = EfficientNetNativeCAM(spec).to(device)
    selected_checkpoint = load_checkpoint(
        entry["directory"] / "best_model.pth"
    )
    model.load_state_dict(selected_checkpoint["model_state_dict"])
    model.eval()
    rows = []
    for index in tqdm.tqdm(audit_indices, desc=f"CAM audit {spec['name']}"):
        tensor, true_grade, path = validation_dataset[index]
        image = tensor.to(device)
        with torch.no_grad():
            logits, maps = model(image[None]), model.class_maps(image[None])
            predicted_grade = int(logits.argmax(1).item())
            cam = model.native_cam_from_maps(
                maps, predicted_grade, image.shape[-2:]
            ).cpu().numpy()
        rows.append({
            "path": path,
            "true_grade": int(true_grade),
            "predicted_grade": predicted_grade,
            **cam_geometry(cam),
            **occlusion_metrics(
                model, image, predicted_grade, cam, spec["physical_batch_size"]
            ),
        })
    with open(entry["directory"] / "native_cam_audit.csv", "w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)
    summary = {
        "variant": spec["name"],
        "audited_cases": len(rows),
        "source_resolution": 12,
        "joint_energy": float(np.mean([row["joint_energy"] for row in rows])),
        "border_energy": float(np.mean([row["border_energy"] for row in rows])),
        "lower_tibia_energy": float(np.mean([row["lower_tibia_energy"] for row in rows])),
        "peak_inside_joint_rate": float(np.mean([row["peak_inside_joint"] for row in rows])),
        "occlusion_correlation": float(np.mean([row["occlusion_correlation"] for row in rows])),
        "top_cam_confidence_drop": float(np.mean([row["top_cam_confidence_drop"] for row in rows])),
    }
    with open(entry["directory"] / "native_cam_summary.json", "w") as handle:
        json.dump(summary, handle, indent=2)
    entry["manifest"]["native_cam_summary"] = summary
    with open(entry["directory"] / "run_manifest.json", "w") as handle:
        json.dump(entry["manifest"], handle, indent=2)
    cam_summaries.append(summary)
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

cam_frame = __import__("pandas").DataFrame(cam_summaries)


## Compare Validation Metrics and CAM Quality

This table is the decision aid, not an automatic production promotion. Prefer a
smaller model when validation selection and CAM faithfulness are comparable; prefer
a larger model only when it produces a meaningful validation gain without worse
joint/border localization.


In [ ]:
metric_rows = []
for variant_name, entry in trained.items():
    metric_rows.append({
        "variant": variant_name,
        "best_epoch": entry["manifest"]["best_epoch"],
        "elapsed_minutes": entry["manifest"]["elapsed_minutes"],
        **entry["manifest"]["validation_metrics"],
    })
metric_frame = __import__("pandas").DataFrame(metric_rows)
comparison = metric_frame.merge(cam_frame, on="variant", how="inner")
comparison = comparison.sort_values("selection_score", ascending=False)
comparison.to_csv(RUN_ROOT / "variant_comparison.csv", index=False)
comparison.to_json(RUN_ROOT / "variant_comparison.json", orient="records", indent=2)
display(comparison[[
    "variant", "best_epoch", "selection_score", "qwk", "macro_f1",
    "grade1_recall", "joint_energy", "border_energy",
    "peak_inside_joint_rate", "occlusion_correlation", "elapsed_minutes",
]].round(4))

winner = comparison.iloc[0].to_dict()
winner_name = str(winner["variant"])
(RUN_ROOT / "VALIDATION_WINNER.txt").write_text(
    winner_name + "\nRun its standalone production notebook before locked test evaluation.\n"
)

report = f'''# EfficientNet Scale Ablation Report

| Field | Value |
| --- | --- |
| Exact UTC run timestamp | {RUN_TIMESTAMP} |
| Variants | B0, B1, B2, B3, B4 |
| Data split | Validation only; test was not read |
| Audit | Up to {CAM_AUDIT_PER_GRADE} cases per grade per variant |
| Selection | QWK + macro F1 + Grade 1 recall + AP/AUC composite |
| Validation winner | `{winner_name}` |

## Comparison

| Variant | Best epoch | Selection | QWK | Macro F1 | Grade 1 recall | Joint energy | Border energy | Peak inside | Occlusion correlation |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
'''
for _, row in comparison.iterrows():
    report += (
        f"| {row['variant']} | {int(row['best_epoch'])} | "
        f"{row['selection_score']:.4f} | {row['qwk']:.4f} | "
        f"{row['macro_f1']:.4f} | {row['grade1_recall']:.4f} | "
        f"{row['joint_energy']:.4f} | {row['border_energy']:.4f} | "
        f"{row['peak_inside_joint_rate']:.4f} | "
        f"{row['occlusion_correlation']:.4f} |\n"
    )
report += (
    "\nThe validation winner is not automatically production-promoted. Copy the "
    "winner's checkpoint into the standalone EfficientNet notebook configuration, "
    "run its locked test evaluation once, and then compare its test metrics and 50-per-grade "
    "CAM report against DenseNet-121 and SE-ResNeXt.\n"
)
(RUN_ROOT / "report.md").write_text(report, encoding="utf-8")
print(report)
print(f"All ablation artifacts: {RUN_ROOT}")


In [ ]:
try:
    from google.colab import runtime
    print("All artifacts saved. Releasing the Colab runtime.")
    runtime.unassign()
except ImportError:
    print("Not running in Colab; runtime release skipped.")
